In [ ]:
# Keep Kaggle output readable: constant genes are ranked last by the F-test implementation.
import warnings
warnings.filterwarnings('ignore', message=r'Features .* are constant\.', category=UserWarning)
warnings.filterwarnings('ignore', message=r'invalid value encountered in divide', category=RuntimeWarning)


# E2 + E3 v3 — sharded runner

Run one `SESSION_ID` per Kaggle session. Each completed configuration is checkpointed immediately.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
for variable in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'
INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
COHORT = 'tcga_luad_lusc_primary_tumor'
SESSION_METHODS = {
    's1_filters': ('f_test','mutual_info','mrmr'),
    's2_embedded_rf': ('l1_lr','elastic_net','rf_importance'),
    's3_relieff': ('relieff',),
    's4_svm_rfe': ('svm_rfe',),
    's5_ga': ('ga',),
    's6_bpso': ('bpso',),
}
SESSION_ID = 's1_filters'  # change for each of the six sessions
assert SESSION_ID in SESSION_METHODS
METHODS_TO_RUN = SESSION_METHODS[SESSION_ID]
E2_K_GRID = [10,50]
E2_REPEATS = 3
E3_K, E3_N_GRID, E3_P_GRID = 50, [100,200,300,None], [1000,2000,5000,10000,None]
E3_WRAPPER_BUDGETS = [(10,20),(30,50)]


In [ ]:
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
# Do not install requirements.txt: preserve Kaggle's compatible scientific stack.
RUNTIME_IMPORTS = ('numpy','pandas','scipy','sklearn','tables','joblib','threadpoolctl','psutil')
for package in RUNTIME_IMPORTS: print(package, __import__(package).__version__)
try:
    import skrebate
except ModuleNotFoundError:
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','skrebate==0.8.4'],check=True)
WORK_ROOT.mkdir(parents=True,exist_ok=True)
for folder in ('experiment-data-perm','experiment-data-geo-target','split_manifests'):
    source,target=INPUT_V3_DATASET/folder,WORK_ROOT/folder
    assert source.is_dir(),source
    if not target.exists(): target.symlink_to(source,target_is_directory=True)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()


In [ ]:
os.chdir(REPO_DIR); sys.path.insert(0,str(REPO_DIR))
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.data import load_dataset, sha256, variance_prefilter
from survey_v2.e1_splits import validate_repeated_split_manifest
from survey_v2.methods import select
from survey_v2.timing import empirical_exponent, machine_info, measure, subsample_rows, top_variance_columns
from scripts.run_e1_feature_selection import split_indices

def atomic_csv(frame,path):
    path.parent.mkdir(parents=True,exist_ok=True); temp=path.with_suffix(path.suffix+'.tmp'); frame.to_csv(temp,index=False); temp.replace(path)
def timed_selection(method,X,y,k,seed,wrapper_kwargs=None):
    def run(X_arg,y_arg,k_arg):
        with threadpool_limits(1): return select(method,X_arg,y_arg,[k_arg],seed=seed,n_jobs=1,wrapper_kwargs=wrapper_kwargs)
    return measure(run,X,y,k,repeats=E2_REPEATS)
def checkpoint(kind,key,row):
    path=checkpoint_root/kind/f'{key}.csv'
    if path.is_file(): return pd.read_csv(path).iloc[0].to_dict()
    atomic_csv(pd.DataFrame([row]),path); return row

ds=load_dataset(WORK_ROOT,COHORT)
manifest_path=ds.data_dir/'repeated_splits.csv'; manifest=pd.read_csv(manifest_path)
errors=validate_repeated_split_manifest(manifest,n_folds=C.N_FOLDS); assert not errors,errors
train,_=split_indices(ds,manifest,repeat=0,fold=0)
seed=int(manifest[(manifest.repeat==0)&(manifest.fold==0)].seed.iloc[0])
X_train,y_train=ds.X[train],ds.y[train]
output_root=WORK_ROOT/'results'/'survey_v3'/'E2_E3'/'shards'/SESSION_ID
checkpoint_root=output_root/'checkpoints'
(output_root/'machine_config.json').parent.mkdir(parents=True,exist_ok=True)
(output_root/'machine_config.json').write_text(json.dumps(machine_info(),indent=2),encoding='utf-8')
pre=variance_prefilter(X_train,5000)


In [ ]:
e2_rows=[]; n_rows=[]; p_rows=[]; wrapper_rows=[]
for method_id,method in enumerate(METHODS_TO_RUN):
    is_wrapper=method in {'ga','bpso'}
    for k in E2_K_GRID:
        budget={'pop':30,'generations':50} if is_wrapper else None
        result=timed_selection(method,X_train[:,pre],y_train,k,C.seed_for(C.dataset_seed_id(COHORT),seed,0,method_id,k),budget)
        row={'experiment':'E2','cohort':COHORT,'method':method,'k':k,'n':len(train),'p':len(pre),'wrapper_population':30 if is_wrapper else None,'wrapper_generations':50 if is_wrapper else None,**result}
        e2_rows.append(checkpoint('e2',f'{method}_k{k}',row)); print('E2',method,k,flush=True)
    if not is_wrapper:
        for n in E3_N_GRID:
            idx=subsample_rows(y_train,n,np.random.default_rng([C.dataset_seed_id(COHORT),seed,3,0 if n is None else n]))
            cols=variance_prefilter(X_train[idx],5000)
            result=timed_selection(method,X_train[idx][:,cols],y_train[idx],E3_K,C.seed_for(C.dataset_seed_id(COHORT),seed,3,method_id,len(idx)))
            row={'cohort':COHORT,'method':method,'k':E3_K,'n':len(idx),'p':len(cols),**result}
            n_rows.append(checkpoint('e3_n',f'{method}_n{len(idx)}',row))
        for p in E3_P_GRID:
            cols=top_variance_columns(X_train,p)
            result=timed_selection(method,X_train[:,cols],y_train,E3_K,C.seed_for(C.dataset_seed_id(COHORT),seed,4,method_id,len(cols)))
            row={'cohort':COHORT,'method':method,'k':E3_K,'n':len(train),'p':len(cols),**result}
            p_rows.append(checkpoint('e3_p',f'{method}_p{len(cols)}',row))
    else:
        for pop,generations in E3_WRAPPER_BUDGETS:
            result=timed_selection(method,X_train[:,pre],y_train,E3_K,C.seed_for(C.dataset_seed_id(COHORT),seed,5,method_id,pop,generations),{'pop':pop,'generations':generations})
            row={'cohort':COHORT,'method':method,'k':E3_K,'n':len(train),'p':len(pre),'population':pop,'generations':generations,'expected_evaluations':pop*generations,**result}
            wrapper_rows.append(checkpoint('e3_wrapper',f'{method}_p{pop}_g{generations}',row))
atomic_csv(pd.DataFrame(e2_rows),output_root/'e2_cost_point.csv')
if n_rows: atomic_csv(pd.DataFrame(n_rows),output_root/'e3_scaling_n.csv')
if p_rows: atomic_csv(pd.DataFrame(p_rows),output_root/'e3_scaling_p.csv')
if wrapper_rows: atomic_csv(pd.DataFrame(wrapper_rows),output_root/'e3_wrapper_evals.csv')


In [ ]:
if n_rows and p_rows:
    n_df,p_df=pd.DataFrame(n_rows),pd.DataFrame(p_rows)
    exponents=[{'cohort':COHORT,'method':method,'n_exponent':empirical_exponent(n_df[n_df.method==method],'n'),'p_exponent':empirical_exponent(p_df[p_df.method==method],'p')} for method in METHODS_TO_RUN]
    atomic_csv(pd.DataFrame(exponents),output_root/'e3_exponents.csv')
config={'experiment':'E2_E3_v3_shard','session_id':SESSION_ID,'methods':METHODS_TO_RUN,'cohort':COHORT,'outer_train_split':{'repeat':0,'fold':0,'split_sha256':sha256(manifest_path)},'e2_repeats':E2_REPEATS,'blas_threads':1,'commit_hash':COMMIT,'traceable':True}
(output_root/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
archive=Path('/kaggle/working')/f'e2_e3_{SESSION_ID}.zip'
shutil.make_archive(str(archive.with_suffix('')),'zip',root_dir=WORK_ROOT,base_dir=str(Path('results/survey_v3/E2_E3/shards')/SESSION_ID))
print('DONE:',archive)


## Merge six completed shard ZIPs

After all six shard sessions finish, attach their six ZIP files as Kaggle inputs. Run only the cell below (plus any imports it contains); do not rerun the measurement cells.


In [ ]:
# Fill these six exact Kaggle input paths, one for each e2_e3_<session>.zip.
SHARD_ARCHIVES = []
assert len(SHARD_ARCHIVES)==6, 'Provide the six completed shard ZIP paths.'
import zipfile
import numpy as np
import pandas as pd
merge_root=Path('/kaggle/working/e2e3_merge')
if merge_root.exists(): shutil.rmtree(merge_root)
merge_root.mkdir(parents=True)
sources=[]
for i,item in enumerate(SHARD_ARCHIVES):
    destination=merge_root/f'input_{i}'
    with zipfile.ZipFile(item) as archive: archive.extractall(destination)
    configs=list(destination.rglob('config.json')); assert len(configs)==1,(item,configs)
    sources.append(configs[0].parent)
expected={'f_test','mutual_info','mrmr','l1_lr','elastic_net','rf_importance','relieff','svm_rfe','ga','bpso'}
seen=set().union(*(set(json.loads((source/'config.json').read_text())['methods']) for source in sources))
assert seen==expected,(seen,expected)
output=merge_root/'E2_E3'; output.mkdir()
def combine(name):
    frames=[pd.read_csv(source/name) for source in sources if (source/name).is_file()]
    if frames: pd.concat(frames,ignore_index=True).to_csv(output/name,index=False)
combine('e2_cost_point.csv'); combine('e3_scaling_n.csv'); combine('e3_scaling_p.csv'); combine('e3_wrapper_evals.csv')
def empirical_exponent(frame,column):
    frame=frame[[column,'wall_s']].dropna()
    if len(frame)<2 or frame[column].nunique()<2: return float('nan')
    return float(np.polyfit(np.log(frame[column]),np.log(frame.wall_s),1)[0])
n_df,p_df=pd.read_csv(output/'e3_scaling_n.csv'),pd.read_csv(output/'e3_scaling_p.csv')
rows=[{'cohort':'tcga_luad_lusc_primary_tumor','method':method,'n_exponent':empirical_exponent(n_df[n_df.method==method],'n'),'p_exponent':empirical_exponent(p_df[p_df.method==method],'p')} for method in sorted(set(n_df.method))]
pd.DataFrame(rows).to_csv(output/'e3_exponents.csv',index=False)
(output/'config.json').write_text(json.dumps({'experiment':'E2_E3_v3_merged','shard_archives':[str(x) for x in SHARD_ARCHIVES],'methods':sorted(seen)},indent=2),encoding='utf-8')
archive=shutil.make_archive('/kaggle/working/e2_e3_merged','zip',merge_root,'E2_E3')
print('DONE:',archive)
